# Day 39: Semantic Routing for Agentic AI

Welcome to Day 39 of the AI Engineering Mastery program! Today, we explore how to implement a **Semantic Router** to classify user intent and route queries to specialized agents.

## Core Theory: The "Why" and "How"

As we transition into building multi-agent systems, treating every user query the exact same way is inefficient. 

**Why Semantic Routing?**
- **Efficiency:** Not every query requires an expensive, multi-step ReAct (Reasoning and Acting) loop. Simple queries can be answered quickly, while complex analytical questions are routed to heavy-duty data agents.
- **Specialization:** Agents perform best when they have a narrow, well-defined scope (specific prompts and specific tools). Routing allows you to split the problem space into distinct domains.
- **Security/Control:** You can route out-of-domain or potentially harmful queries to a safety fallback agent before they touch your core business logic.

**How does it work?**
A Semantic Router sits at the very entry point of your architecture. Instead of rigid keyword matching or regex, it leverages LLMs (via structured output/function calling) or Embedding comparisons to dynamically classify the semantic intent of the input text. 

Today, we will implement an LLM-based router using LangChain and Pydantic. By forcing the LLM to output a strict JSON schema, we guarantee that the output perfectly matches our predefined routing categories.

## Common Pitfalls in Production
1. **Overlapping Intents:** If your categories are too similar (e.g., `sales_agent` vs `pricing_agent`), the router will struggle and lower its confidence. Keep intents mutually exclusive and distinct.
2. **Latency Overhead:** Using a massive, slow model (like GPT-4) just to route a query adds unnecessary latency. Use faster, smaller models (like GPT-3.5-Turbo or Claude 3 Haiku) specifically tuned for the routing step.
3. **Missing Fallbacks:** Users *will* ask things your system wasn't designed for. Always have a fallback mechanism for low-confidence scores or unrecognized intents.

## Reference Links
- [LangChain Structured Output](https://python.langchain.com/docs/how_to/structured_output/)
- [Pydantic Official Documentation](https://docs.pydantic.dev/latest/)
- [OpenAI Function Calling & Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs)


## Code Implementation: Building the Router

Below is a tiered progression of Python code examples demonstrating how to build a Semantic Router, from a basic proof-of-concept to a production-ready system.

In [ ]:
# --- BASIC IMPLEMENTATION ---
# Isolate the core concept with minimal boilerplate.
# This uses simple zero-shot classification via LangChain prompt.

import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# Ensure API key is present or handled safely
api_key = os.environ.get("OPENAI_API_KEY", "sk-dummy-key")

def basic_semantic_router(query: str) -> str:
    """A basic router using zero-shot prompt classification."""
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0, api_key=api_key)
    
    prompt = PromptTemplate.from_template(
        "Classify the following query into exactly one of these categories: "
        "[support, analytics, general].\n\nQuery: {query}\n\nCategory:"
    )
    
    chain = prompt | llm
    try:
        response = chain.invoke({"query": query})
        return response.content.strip().lower()
    except Exception as e:
        return f"Error: {e}"

if __name__ == "__main__":
    print("Basic Router Output:", basic_semantic_router("My account is locked!"))


In [ ]:
# --- MEDIUM IMPLEMENTATION ---
# Emphasize clean OOP, state management, and structured output.

import os
from enum import Enum
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

class Intent(str, Enum):
    SUPPORT = "support"
    ANALYTICS = "analytics"
    GENERAL = "general"

class RouteResult(BaseModel):
    intent: Intent = Field(description="The primary intent of the user query.")

class SemanticRouter:
    def __init__(self, model_name: str = "gpt-3.5-turbo"):
        self.api_key = os.environ.get("OPENAI_API_KEY", "sk-dummy-key")
        self.llm = ChatOpenAI(model=model_name, temperature=0.0, api_key=self.api_key)
        
        # Bind the Pydantic model to force structured JSON output
        self.structured_llm = self.llm.with_structured_output(RouteResult)
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an expert router. Classify the user query into support, analytics, or general."),
            ("user", "{query}")
        ])
        self.chain = self.prompt | self.structured_llm

    def route(self, query: str) -> str:
        try:
            result = self.chain.invoke({"query": query})
            return result.intent.value
        except Exception as e:
            return f"fallback_general (Error: {e})"

if __name__ == "__main__":
    router = SemanticRouter()
    print("Medium Router Output:", router.route("Show me the sales data from last month."))


In [ ]:
# --- ADVANCED IMPLEMENTATION ---
# Production-grade implementation with strict type hinting, docstrings, 
# error handling, AI Security (fallback mechanisms), and exact import syntax.

import os
import logging
from enum import Enum
from typing import Optional, Dict, Callable
from pydantic import BaseModel, Field, ValidationError
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import Runnable

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class RouteIntent(str, Enum):
    """Supported routing destinations for our system."""
    CUSTOMER_SUPPORT = "customer_support"
    DATA_ANALYTICS = "data_analytics"
    TECHNICAL_DOCS = "technical_docs"
    GENERAL_CHAT = "general_chat"
    FALLBACK = "fallback"  # AI Security: Always have a fallback intent

class RouterOutput(BaseModel):
    """Structured output schema enforced on the LLM."""
    intent: RouteIntent = Field(
        description="The primary intent of the user query."
    )
    confidence_score: float = Field(
        description="Confidence in the classification between 0.0 and 1.0."
    )
    reasoning: str = Field(
        description="Brief explanation for why this intent was chosen."
    )

class ProductionSemanticRouter:
    """
    Production-grade Semantic Router that classifies intents and routes queries
    to appropriate agent functions with robust fallback mechanisms.
    """
    def __init__(self, model_name: str = "gpt-3.5-turbo", confidence_threshold: float = 0.75):
        self.model_name = model_name
        self.confidence_threshold = confidence_threshold
        
        # AI Security: Strict API Key retrieval - No hardcoded fallback secrets in os.getenv()
        self.api_key = os.environ.get("OPENAI_API_KEY")
        if not self.api_key:
            logger.warning("OPENAI_API_KEY not found. Using dummy key for testing.")
            self.api_key = "sk-dummy-key"
            
        self.llm = ChatOpenAI(model=self.model_name, temperature=0.0, api_key=self.api_key)
        self.chain = self._build_chain()
        self.routes: Dict[RouteIntent, Callable[[str], str]] = {}

    def _build_chain(self) -> Runnable:
        """Constructs the LCEL chain for structured intent classification."""
        structured_llm = self.llm.with_structured_output(RouterOutput)
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an expert intent classification router. "
                       "Analyze the user's query and classify it into one of the exact specified intents. "
                       "Ensure your reasoning does not regurgitate PII (Personally Identifiable Information). "
                       "If the query doesn't strongly fit any category, lower your confidence score."),
            ("user", "Query: {query}")
        ])
        return prompt | structured_llm

    def register_route(self, intent: RouteIntent, agent_func: Callable[[str], str]) -> None:
        """Registers a handler function for a specific intent."""
        self.routes[intent] = agent_func

    def route_query(self, query: str) -> str:
        """
        Classifies the query and executes the appropriate agent function,
        falling back if confidence is too low or an error occurs.
        """
        try:
            logger.info(f"Classifying query: '{query[:50]}...'")
            result: RouterOutput = self.chain.invoke({"query": query})
            logger.info(f"Intent: {result.intent.name}, Confidence: {result.confidence_score:.2f}")
            
            if result.confidence_score < self.confidence_threshold:
                logger.warning("Confidence below threshold. Routing to FALLBACK.")
                intent_to_execute = RouteIntent.FALLBACK
            else:
                intent_to_execute = result.intent
                
        except ValidationError as ve:
            logger.error(f"Output validation failed: {ve}")
            intent_to_execute = RouteIntent.FALLBACK
        except Exception as e:
            logger.error(f"Routing failed due to API or network error: {e}")
            intent_to_execute = RouteIntent.FALLBACK

        agent_func = self.routes.get(intent_to_execute, self.routes.get(RouteIntent.FALLBACK))
        if agent_func:
            return agent_func(query)
        return "Critical Error: No fallback agent registered."

# Mock Destination Agents
def support_agent(query: str) -> str: return "[Support Agent] Handling ticket."
def analytics_agent(query: str) -> str: return "[Analytics Agent] Executing analysis."
def general_agent(query: str) -> str: return "[General Agent] Handling chat."
def fallback_agent(query: str) -> str: return "[Fallback Agent] I couldn't process that safely."

if __name__ == "__main__":
    router = ProductionSemanticRouter()
    router.register_route(RouteIntent.CUSTOMER_SUPPORT, support_agent)
    router.register_route(RouteIntent.DATA_ANALYTICS, analytics_agent)
    router.register_route(RouteIntent.GENERAL_CHAT, general_agent)
    router.register_route(RouteIntent.FALLBACK, fallback_agent)
    
    print("Advanced Router Output:", router.route_query("My account is locked and I can't log in!"))


## Practical Lab / Homework

**Your Task:** 
In the **Advanced** implementation above, we built a production-ready semantic router with a fallback mechanism. Now it's your turn to extend it by adding a **PII Redaction pre-processing step** and a new agent to adhere to AI Security best practices.

1. Implement a robust `redact_pii(query: str) -> str` function that uses regex to replace email addresses and standard phone numbers with `[REDACTED]`. (Do not use hardcoded string replacements).
2. Create an extended `SecureSemanticRouter` class that wraps the `ProductionSemanticRouter` to ensure the query is sanitized *before* being sent to the LLM for classification.
3. Add a new intent `RouteIntent.SALES` to the system and register a `sales_agent` to handle queries about pricing and purchasing.
4. Test your implementation by passing a query containing an email address and asking for pricing details.
5. **Bonus:** Record a brief async video walkthrough (using Loom or similar) explaining your design decisions, specifically focusing on why PII redaction should happen before routing.

In [ ]:
# --- LAB IMPLEMENTATION ---
# Fully implemented, robust logic for PII redaction and secure routing.

import re
import os
import logging
from enum import Enum
from typing import Dict, Callable
from pydantic import BaseModel, Field, ValidationError
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import Runnable

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# 1. Extend the Intent Enum
class ExtendedRouteIntent(str, Enum):
    CUSTOMER_SUPPORT = "customer_support"
    DATA_ANALYTICS = "data_analytics"
    TECHNICAL_DOCS = "technical_docs"
    GENERAL_CHAT = "general_chat"
    SALES = "sales"
    FALLBACK = "fallback"

class ExtendedRouterOutput(BaseModel):
    intent: ExtendedRouteIntent = Field(description="The primary intent of the user query.")
    confidence_score: float = Field(description="Confidence in the classification between 0.0 and 1.0.")
    reasoning: str = Field(description="Brief explanation for why this intent was chosen.")

# 2. PII Redaction Function
def redact_pii(query: str) -> str:
    """Replaces email addresses and phone numbers with [REDACTED] using robust regex."""
    # Regex for email
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    # Regex for standard phone numbers (e.g., 123-456-7890, (123) 456-7890)
    phone_pattern = r'\b(?:\+\d{1,2}\s)?\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}\b'
    
    sanitized = re.sub(email_pattern, '[REDACTED_EMAIL]', query)
    sanitized = re.sub(phone_pattern, '[REDACTED_PHONE]', sanitized)
    return sanitized

# 3. Secure Semantic Router incorporating PII redaction and new intent
class SecureSemanticRouter:
    def __init__(self, model_name: str = "gpt-3.5-turbo", confidence_threshold: float = 0.75):
        self.model_name = model_name
        self.confidence_threshold = confidence_threshold
        
        self.api_key = os.environ.get("OPENAI_API_KEY")
        if not self.api_key:
            logger.warning("OPENAI_API_KEY not found. Using dummy key for testing.")
            self.api_key = "sk-dummy-key"
            
        self.llm = ChatOpenAI(model=self.model_name, temperature=0.0, api_key=self.api_key)
        self.chain = self._build_chain()
        self.routes: Dict[ExtendedRouteIntent, Callable[[str], str]] = {}

    def _build_chain(self) -> Runnable:
        structured_llm = self.llm.with_structured_output(ExtendedRouterOutput)
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an expert intent classification router. "
                       "Classify the user's query into one of the exact specified intents. "
                       "If it's about pricing or purchasing, choose 'sales'."),
            ("user", "Query: {query}")
        ])
        return prompt | structured_llm

    def register_route(self, intent: ExtendedRouteIntent, agent_func: Callable[[str], str]) -> None:
        self.routes[intent] = agent_func

    def route_query(self, original_query: str) -> str:
        # AI Security: Sanitize PII before sending to LLM
        sanitized_query = redact_pii(original_query)
        logger.info(f"Sanitized query being routed: '{sanitized_query[:50]}...'")
        
        try:
            result: ExtendedRouterOutput = self.chain.invoke({"query": sanitized_query})
            logger.info(f"Intent: {result.intent.name}, Confidence: {result.confidence_score:.2f}")
            
            if result.confidence_score < self.confidence_threshold:
                intent_to_execute = ExtendedRouteIntent.FALLBACK
            else:
                intent_to_execute = result.intent
        except Exception as e:
            logger.error(f"Routing failed: {e}")
            intent_to_execute = ExtendedRouteIntent.FALLBACK

        agent_func = self.routes.get(intent_to_execute, self.routes.get(ExtendedRouteIntent.FALLBACK))
        if agent_func:
            # The agent receives the sanitized query for safety
            return agent_func(sanitized_query)
        return "Critical Error: No fallback registered."

# Agents
def support_agent(query: str) -> str: return f"[Support Agent] Processing: {query}"
def sales_agent(query: str) -> str: return f"[Sales Agent] Processing: {query}"
def fallback_agent(query: str) -> str: return f"[Fallback Agent] Safely handling unrecognized query."

if __name__ == "__main__":
    secure_router = SecureSemanticRouter()
    secure_router.register_route(ExtendedRouteIntent.CUSTOMER_SUPPORT, support_agent)
    secure_router.register_route(ExtendedRouteIntent.SALES, sales_agent)
    secure_router.register_route(ExtendedRouteIntent.FALLBACK, fallback_agent)
    
    # Test with PII and sales intent
    test_query = "Hi, my email is user123@example.com and phone is (555) 123-4567. What is the pricing for the enterprise plan?"
    print("\n--- Lab Execution ---")
    print(secure_router.route_query(test_query))
